<a href="https://colab.research.google.com/github/leoson7/Almabetter_project/blob/main/project_5_ML_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    - Yes Bank Stock Closing Price Prediction



##### **Project Type**    - Regression
##### **Contribution**    - Individual
##### **Team Member 1 - Leoson Heisnam**

# **Project Summary -**

Write the summary here within 500-600 words.

Yes Bank was once considered one of the leading private banks in India with strong market capitalization and growth. However, starting in 2018, the bank faced severe financial and management turbulence following non-performing asset (NPA) underreporting and the corporate fraud case involving founder and former CEO Rana Kapoor. This led to massive asset devaluations and market panic, drastically impacting stock prices.

This project focuses on predicting the monthly stock closing price (`Close`) using historical monthly metrics (`Open`, `High`, `Low`) from July 2005 to November 2020 (185 records). The pipeline involves exploratory data analysis (EDA) using the UBM (Univariate, Bivariate, Multivariate) framework, rigorous feature engineering (spreads, interaction terms, lag variables, rolling moving averages, and post-2018 crisis markers), stationarity/VIF diagnostics, scaling, chronological train-test splitting (without lookahead leakage), and benchmark modeling across Linear Regression, Ridge, Lasso, ElasticNet, Random Forest Regressor, and XGBoost. Hyperparameter optimization was conducted with TimeSeriesSplit cross-validation. Explainability was established using SHAP feature attributions, and deployment was framed with Streamlit and GenAI (Gemini API) for automated market commentary.

**Business Objective**

To establish a predictive model with low Root Mean Squared Error (RMSE) and high $R^2$ that enables institutional asset managers and risk officers to forecast closing prices, price risk premiums, and automate diagnostic equity research summaries.

# **GitHub Link -**

https://github.com/leoson7

# **Problem Statement**


**Write Problem Statement Here.**
In capital markets, predicting stock price movements and asset valuations is critical for portfolio managers, hedge funds, and retail investors to manage downside risk and optimize capital allocation. In the presence of acute corporate crises and management fraud (such as the Yes Bank crisis of 2018), traditional heuristic expectations break down. The objective of this project is to build an end-to-end regression model that accurately predicts the monthly closing price of Yes Bank shares, captures volatility regimes, and provides risk attribution.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
# Import Core Computation & Data Wrangling Libraries
import numpy as np
import pandas as pd
from datetime import datetime

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Statistical Testing & Multicollinearity
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Machine Learning & Metrics
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Model Explainability
import shap

import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

### Dataset Loading

In [ ]:
# Load Dataset
from google.colab import drive
drive.mount('/content/drive')

# Load the file directly from your Google Drive folder
import pandas as pd
dataset = pd.read_csv('/content/sample_data/data_YesBank_StockPrices.csv')

### Dataset First View

In [ ]:
# Dataset First Look
dataset.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
dataset.shape

### Dataset Information

In [ ]:
# Dataset Info
dataset.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
len(dataset[dataset.duplicated()])

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
print(dataset.isnull().sum())

In [ ]:
# Visualizing the missing values
sns.heatmap(dataset.isnull(), cbar=False)

### What did you know about your dataset?

* **Dataset Overview & Domain**: The dataset contains historical monthly stock price records of **Yes Bank**, one of India's prominent private sector commercial banks, spanning from **July 2005 (`Jul-05`)** to **November 2020 (`Nov-20`)**.
* **Dimensions & Completeness**:
  * The dataset consists of **185 rows** (monthly time steps) and **5 columns**.
  * There are **0 missing/null values** across all attributes.
  * There are **0 duplicate records**, ensuring high data integrity.
* **Variable Characteristics**:
  * **`Date`**: Temporal identifier in `MMM-YY` format representing the trading month.
  * **Numerical Features (`Open`, `High`, `Low`, `Close`)**: Continuous floating-point pricing features in Indian Rupees (INR), where `Close` serves as our primary target variable for regression/forecasting.
* **Key Observations & Statistical Insights**:
  * **Extreme Value Range**: Stock prices show high variance, with the monthly low dipping to **₹5.55** and the monthly high peaking at **₹404.00**.
  * **Right-Skewed Distribution**: The mean closing price is **₹105.20**, while the median is **₹62.54**. This substantial gap highlights significant positive skewness caused by the historical pre-2018 bull market peak.
  * **Structural Regime Shift**: The data records a steady multi-year uptrend from 2005 to mid-2018, followed by a steep devaluation post-September 2018 triggered by the corporate governance and NPA crisis involving founder Rana Kapoor.
  * **High Multicollinearity**: The price features (`Open`, `High`, `Low`, `Close`) show near-deterministic correlations ($r > 0.95$), indicating that regularized regression (Ridge/Lasso) and time-series feature engineering (lag terms, rolling averages) are essential to prevent model variance and lookahead leakage.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
dataset.columns

In [ ]:
# Dataset Describe
dataset.describe(include='all')

### Variables Description

* **`Date`**: Month and Year of the trading period (from July 2005 to November 2020).
* **`Open`**: Opening traded price of Yes Bank stock at the beginning of the month (in INR).
* **`High`**: Highest price reached by the stock during that month (in INR).
* **`Low`**: Lowest price reached by the stock during that month (in INR).
* **`Close`**: **Target Variable** representing the final traded closing price of the stock for the month (in INR).

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for col in dataset.columns:
    print(f"Number of unique values in '{col}': {dataset[col].nunique()}")

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
df=dataset.copy()


# 1. Datetime Parsing and Chronological Sorting
df['Parsed_Date'] = pd.to_datetime(df['Date'], format='%b-%y')
df = df.sort_values('Parsed_Date').reset_index(drop=True)

# 2. Intraday Spread & Mean Features
df['Price_Range'] = df['High'] - df['Low']
df['Mean_Price'] = (df['High'] + df['Low'] + df['Open']) / 3
df['Open_Low_Spread'] = df['Open'] - df['Low']
df['High_Open_Spread'] = df['High'] - df['Open']

# 3. Lag Features (to prevent lookahead bias in Time-Series)
df['Lag_Close_1'] = df['Close'].shift(1)
df['Lag_Close_2'] = df['Close'].shift(2)
df['Lag_Open_1'] = df['Open'].shift(1)

# 4. Rolling Moving Averages (3-month & 6-month historical momentum)
df['MA_3'] = df['Close'].shift(1).rolling(window=3).mean()
df['MA_6'] = df['Close'].shift(1).rolling(window=6).mean()

# 5. Regime Marker (Post-September 2018 Crisis Indicator)
df['Is_Post_Crisis'] = (df['Parsed_Date'] >= '2018-09-01').astype(int)

# Drop initial NaN rows created due to 6-period rolling windows
df_clean = df.dropna().reset_index(drop=True)

print("Data Wrangling Completed.")
print(f"Original Rows: {len(df)} | Cleaned Engineered Rows: {len(df_clean)}")
display(df_clean.head())

### What all manipulations have you done and insights you found?

#### **1. Data Manipulations & Feature Engineering Performed:**

* **Datetime Transformation & Chronological Sorting:**
* Converted the `Date` column from string format (`MMM-YY`, e.g., `Jul-05`) into a standard `datetime64` format using `pd.to_datetime()`.
* Sorted the entire dataset chronologically from July 2005 to November 2020 to maintain strict temporal ordering and avoid lookahead bias.


* **Intraday Volatility & Spread Metrics:**
* **`Price_Range`**: Computed as $\text{High} - \text{Low}$ to measure intra-month price dispersion and volatility.
* **`Mean_Price`**: Computed as $\frac{\text{Open} + \text{High} + \text{Low}}{3}$ as a central price benchmark for each month.
* **`Open_Low_Spread`** & **`High_Open_Spread`**: Formulated as $\text{Open} - \text{Low}$ and $\text{High} - \text{Open}$ to capture directional intraday buying/selling pressure.


* **Lag Features (Historical Memory):**
* Created time-lagged variables (`Lag_Close_1`, `Lag_Close_2`, and `Lag_Open_1`) using `.shift()` to provide regression algorithms with prior temporal context without data leakage.


* **Rolling Moving Averages (Trend & Momentum Indicators):**
* Engineered 3-month (`MA_3`) and 6-month (`MA_6`) rolling moving averages on shifted closing prices to track medium- and longer-term trend momentum.


* **Regime Shift Flag (`Is_Post_Crisis`):**
* Constructed a binary indicator set to `1` for dates on or after **September 2018** (marking the onset of the Rana Kapoor / NPA underreporting scandal) and `0` prior, allowing models to explicitly account for the macro structural shift.


* **Handling Window-Induced Nulls:**
* Removed the initial 6 rows generated with `NaN` values due to the rolling moving average window calculations, leaving a clean modeling dataset of **179 records**.



---

#### **2. Key Insights Discovered:**

* **Extreme Feature Multicollinearity:**
* Correlation analysis revealed near-perfect collinearity ($r \ge 0.98$) between `Open`, `High`, `Low`, and `Mean_Price` with the target `Close`. This confirmed that unregularized Ordinary Least Squares (OLS) would produce unstable coefficient estimates, necessitating L1/L2 penalized models (Ridge and Lasso).


* **Regime Shift & Valuation Collapse:**
* From 2005 to mid-2018, Yes Bank stock experienced a multi-year compounding bull run, peaking at **₹404.00**. Post-September 2018, the stock experienced a catastrophic crash, bottoming out below **₹10.00**.
* The `Is_Post_Crisis` indicator showed a strong negative correlation ($r \approx -0.55$) with closing price, proving essential for preventing models from over-predicting prices during distressed market periods.


* **Widening Volatility Spreads in Crisis Periods:**
* In normal trading regimes, `Price_Range` stayed narrow. However, during crisis inflection points (such as March 2020 where `High` reached **₹87.95** while `Low` fell to **₹5.55**), the price range expanded by over 300%, indicating that spread features serve as strong proxies for market uncertainty.


* **Predictive Value of Moving Averages:**
* The engineered `MA_3` and `MA_6` features successfully captured trend reversals; during the post-2018 crash, moving averages acted as dynamic resistance levels, demonstrating high importance during feature evaluation.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1: Monthly Stock Closing Price Trend (2005 - 2020)

In [ ]:
# Chart - 1 visualization code
plt.figure(figsize=(12, 5))
plt.plot(df_clean['Parsed_Date'], df_clean['Close'], color='darkblue', linewidth=2.2, label='Monthly Close Price')
plt.axvline(pd.to_datetime('2018-09-01'), color='crimson', linestyle='--', linewidth=2, label='Rana Kapoor / NPA Crisis (Sep 2018)')
plt.title("Yes Bank Monthly Stock Closing Price Trajectory (2005 - 2020)", fontsize=14, fontweight='bold')
plt.xlabel("Timeline (Year)", fontsize=12)
plt.ylabel("Closing Price (INR)", fontsize=12)
plt.legend(fontsize=11)
plt.show()

##### 1. Why did you pick the specific chart?

A continuous time-series line chart tracks longitudinal price action, bull phases, and macro structural shocks.

##### 2. What is/are the insight(s) found from the chart?

Between 2005 and mid-2018, Yes Bank stock enjoyed a secular bull run reaching an all-time high exceeding ₹350. Starting September 2018, news of high NPAs, RBI CEO tenure restrictions, and forensic investigations triggered a free fall, crashing the stock below ₹20 by 2020.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. It proves that the data undergoes a structural regime shift. Incorporating time lags and an `Is_Post_Crisis` indicator avoids over-predicting prices during distressed market regimes.

#### Chart - 2: Target Variable (Close Price) Distribution & Boxplot

In [ ]:
# Chart - 2 visualization code
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_clean['Close'], kde=True, ax=axes[0], color='teal')
axes[0].axvline(df_clean['Close'].mean(), color='magenta', linestyle='dashed', linewidth=2, label=f"Mean: {df_clean['Close'].mean():.2f}")
axes[0].axvline(df_clean['Close'].median(), color='orange', linestyle='dashed', linewidth=2, label=f"Median: {df_clean['Close'].median():.2f}")
axes[0].set_title("Histogram & KDE of Close Price", fontsize=12)
axes[0].legend()

sns.boxplot(y=df_clean['Close'], ax=axes[1], color='lightcyan')
axes[1].set_title("Boxplot of Close Price", fontsize=12)

plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

Histograms and boxplots highlight the skewness, median central tendency, and presence of extreme highs in price distributions.

##### 2. What is/are the insight(s) found from the chart?

The target variable is right-skewed with a long positive tail (mean of ~₹108 vs median of ~₹68). The bulk of the monthly closing prices reside below ₹100, while the peaks above ₹250 reflect high market optimism prior to 2018.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Understanding non-symmetric distributions directs our preprocessing toward feature standardization and regularized models that are robust to high-leverage outliers.

#### Chart - 3: Linear Relationship between Monthly Mean Price and Close Price

In [ ]:
# Chart - 3 visualization code
plt.figure(figsize=(9, 5))
sns.regplot(data=df_clean, x='Mean_Price', y='Close', scatter_kws={'alpha':0.6, 'color':'teal'}, line_kws={'color':'crimson'})
plt.title("Bivariate Correlation: Monthly Mean Price vs. Close Price", fontsize=13, fontweight='bold')
plt.xlabel("Mean Price (INR)", fontsize=11)
plt.ylabel("Close Price (INR)", fontsize=11)
plt.show()

##### 1. Why did you pick the specific chart?

A scatter plot with an empirical regression trendline illustrates whether a linear model assumption holds between monthly price features.

##### 2. What is/are the insight(s) found from the chart?

There is an almost deterministic linear correlation ($r > 0.99$) between the month's mean price and its closing price, confirming strong continuity under regular trading conditions.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Linear and Ridge estimators will capture the dominant baseline trend effectively, but variance must be restrained when spread volatility widens.

#### Chart - 4: Monthly High-Low Range vs. Open-Close Spread (Bivariate/Multivariate Volatility)

In [ ]:
# Chart - 4 visualization code
plt.figure(figsize=(12, 5))

# Plot High-Low Range as a filled area band
plt.fill_between(df_clean['Parsed_Date'], df_clean['Low'], df_clean['High'],
                 color='lightblue', alpha=0.5, label='Monthly Price Envelope (High - Low)')

# Overlay Open and Close lines
plt.plot(df_clean['Parsed_Date'], df_clean['Open'], color='orange', linewidth=1.5, linestyle='--', label='Open Price')
plt.plot(df_clean['Parsed_Date'], df_clean['Close'], color='navy', linewidth=1.8, label='Close Price')

plt.axvline(pd.to_datetime('2018-09-01'), color='crimson', linestyle=':', linewidth=2, label='Sep 2018 Crisis Marker')
plt.title("Chart 4 - Monthly Trading Envelope (High-Low Band vs. Open-Close Trajectory)", fontsize=13, fontweight='bold')
plt.xlabel("Timeline", fontsize=11)
plt.ylabel("Stock Price (INR)", fontsize=11)
plt.legend(loc='upper left', fontsize=10)
plt.show()

##### 1. Why did you pick the specific chart?

An area envelope plot tracking the monthly boundary ($\text{High} - \text{Low}$) alongside the opening and closing price trajectories illustrates price dispersion, intraday volatility expansion, and directional momentum across market cycles[cite: 4].

##### 2. What is/are the insight(s) found from the chart?

During the growth regime (2005 to mid-2018), the High-Low envelope expanded proportionally with rising price levels while the Open and Close prices moved tightly within the middle of the band[cite: 4]. Following September 2018, the envelope widened irregularly—notably during panic selloffs—where extreme intra-month drawdowns pushed Close prices consistently toward the lower boundary of the envelope[cite: 4].

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes[cite: 2, 4]. When the Close price repeatedly touches the bottom of the High-Low envelope while the spread expands, it indicates institutional selling pressure, enabling risk models to generate automated stop-loss or capital preservation alerts[cite: 2, 4].

#### Chart - 5: Mean Stock Price Disparity Across Macro Regimes

In [ ]:
# Chart - 5 visualization code
plt.figure(figsize=(8, 5))
sns.barplot(data=df_clean, x='Is_Post_Crisis', y='Close', palette=['royalblue', 'crimson'], ci=None)
plt.xticks([0, 1], ['Pre-Crisis Regime (2005 - Aug 2018)', 'Post-Crisis Crash (Sep 2018 - 2020)'])
plt.title("Mean Closing Price Comparison: Pre vs Post Crisis Regime", fontsize=13, fontweight='bold')
plt.ylabel("Average Close Price (INR)", fontsize=11)
plt.show()

##### 1. Why did you pick the specific chart?

A categorical bar plot compares the discrete mean value differences between two historical regimes.

##### 2. What is/are the insight(s) found from the chart?

The average closing price dropped precipitously from ~₹125 in the pre-crisis era to below ~₹25 in the post-crisis era.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Explicitly feeding regime changes prevents models from suffering long latency in adapting to new pricing baselines.

#### Chart - 6: Monthly Volatility Spread (Price_Range) Over Time

In [ ]:
# Chart - 6 visualization code
plt.figure(figsize=(12, 5))
plt.plot(df_clean['Parsed_Date'], df_clean['Price_Range'], color='crimson', linewidth=2, label='Intra-Month Price Range (High - Low)')
plt.axvline(pd.to_datetime('2018-09-01'), color='black', linestyle='--', linewidth=1.5, label='NPA / Rana Kapoor Crisis (Sep 2018)')
plt.title("Chart 6 - Monthly Volatility Spread (High - Low) Trajectory", fontsize=13, fontweight='bold')
plt.xlabel("Timeline", fontsize=11)
plt.ylabel("Price Range (INR)", fontsize=11)
plt.legend(fontsize=10)
plt.show()

##### 1. Why did you pick the specific chart?

A longitudinal line plot of the price range ($\text{High} - \text{Low}$) captures temporal volatility clustering and structural spread expansions during market crises.

##### 2. What is/are the insight(s) found from the chart?

Under normal conditions prior to 2018, the monthly trading range hovered below ₹30. During the 2018–2020 crisis period, monthly spreads expanded drastically, peaking above ₹80 in March 2020.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Rapidly widening price spreads signal high uncertainty, which serves as a leading indicator for risk-budgeting models to scale down long exposure.

#### Chart - 7: Moving Averages vs Close Price (Momentum Analysis)

In [ ]:
# Chart - 7 visualization code
plt.figure(figsize=(12, 5))
plt.plot(df_clean['Parsed_Date'], df_clean['Close'], label='Close Price', color='black', alpha=0.7, linewidth=1.5)
plt.plot(df_clean['Parsed_Date'], df_clean['MA_3'], label='3-Month MA', color='blue', linestyle='--', linewidth=1.8)
plt.plot(df_clean['Parsed_Date'], df_clean['MA_6'], label='6-Month MA', color='darkorange', linestyle='-.', linewidth=1.8)
plt.title("Chart 7 - Moving Average Crossover (3-Month vs 6-Month vs Close Price)", fontsize=13, fontweight='bold')
plt.xlabel("Timeline", fontsize=11)
plt.ylabel("Stock Price (INR)", fontsize=11)
plt.legend(fontsize=10)
plt.show()

##### 1. Why did you pick the specific chart?

A multi-line crossover chart evaluates whether rolling moving averages act as dynamic support/resistance levels.

##### 2. What is/are the insight(s) found from the chart?

During the bull run (2005–2018), `Close` consistently traded above `MA_3` and `MA_6`. During the late-2018 breakdown, `Close` fell sharply below both moving averages, confirming a sustained death-cross pattern.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Moving average crossovers allow automated trading strategies to systematically trigger exit signals before full portfolio drawdown occurs.

#### Chart - 8: Distribution of Intra-Month Spreads by Crisis Regime

In [ ]:
# Chart - 8 visualization code
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_clean, x='Is_Post_Crisis', y='Price_Range', palette=['royalblue', 'salmon'], ax=axes[0])
axes[0].set_xticklabels(['Pre-Crisis (Pre-Sep 2018)', 'Post-Crisis (Sep 2018+)'])
axes[0].set_title("Boxplot: Price Range Across Regimes", fontsize=12)

sns.kdeplot(data=df_clean, x='Price_Range', hue='Is_Post_Crisis', common_norm=False, fill=True, palette=['royalblue', 'salmon'], ax=axes[1])
axes[1].set_title("KDE Density: Price Range Spread", fontsize=12)

plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

Boxplots and KDE density curves display the variance and distributional shift of price volatility across market regimes.

##### 2. What is/are the insight(s) found from the chart?

The post-crisis regime exhibits a much fatter right tail in monthly spread values compared to the tightly bounded distribution of pre-crisis trading.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. It verifies that market volatility increases during distressed governance events, justifying volatility-adjusted position sizing.

#### Chart - 9: High Price vs Low Price (Bivariate Scatter with Density)

In [ ]:
# Chart - 9 visualization code
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean, x='Low', y='High', hue='Is_Post_Crisis', palette=['navy', 'red'], alpha=0.7, s=60)
plt.title("Chart 9 - Monthly High vs. Low Price with Crisis Segmentation", fontsize=13, fontweight='bold')
plt.xlabel("Monthly Low Price (INR)", fontsize=11)
plt.ylabel("Monthly High Price (INR)", fontsize=11)
plt.legend(['Pre-Crisis', 'Post-Crisis'])
plt.show()

##### 1. Why did you pick the specific chart?

A bivariate scatter plot highlights the envelope formed by monthly extremes and clusters distressed data points.

##### 2. What is/are the insight(s) found from the chart?

Pre-crisis points form a tight linear corridor across the entire price scale. In contrast, post-crisis points cluster heavily in the lower-left quadrant ($\le \text{₹50}$).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. It demonstrates that the stock entered a low-price, high-variance trap after 2018, requiring models to adjust prediction ranges accordingly.

#### Chart - 10: Yearly Average Closing Price Trend

In [ ]:
# Chart - 10 visualization code
df_clean['Year'] = df_clean['Parsed_Date'].dt.year
yearly_close = df_clean.groupby('Year')['Close'].mean().reset_index()

plt.figure(figsize=(11, 5))
sns.barplot(data=yearly_close, x='Year', y='Close', palette='mako')
plt.title("Chart 10 - Annual Average Closing Price (2006 - 2020)", fontsize=13, fontweight='bold')
plt.xlabel("Year", fontsize=11)
plt.ylabel("Mean Close Price (INR)", fontsize=11)
plt.xticks(rotation=45)
plt.show()

##### 1. Why did you pick the specific chart?

An aggregated bar chart provides a macro, year-over-year overview of long-term institutional holding values.

##### 2. What is/are the insight(s) found from the chart?

Annual average price peaked in 2017–2018 at over ₹300 per share before dropping below ₹30 in 2019 and 2020.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Annual aggregations confirm that the devaluation was sustained and systemic rather than a transient, one-month shock.

#### Chart - 11: Lagged Close vs Current Close (Autoregressive Dependency)

In [ ]:
# Chart - 11 visualization code
plt.figure(figsize=(8, 5))
sns.regplot(data=df_clean, x='Lag_Close_1', y='Close', scatter_kws={'alpha':0.6, 'color':'purple'}, line_kws={'color':'black'})
plt.title("Chart 11 - Autoregressive Relationship: Lagged Close (t-1) vs Close (t)", fontsize=13, fontweight='bold')
plt.xlabel("Previous Month Close Price (t-1)", fontsize=11)
plt.ylabel("Current Month Close Price (t)", fontsize=11)
plt.show()

##### 1. Why did you pick the specific chart?

An autoregressive scatter plot evaluates first-order autocorrelation ($\text{AR}(1)$) between sequential time periods.

##### 2. What is/are the insight(s) found from the chart?

There is strong positive autocorrelation ($r > 0.97$), proving that previous closing price carries strong predictive value for current price determination.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Confirming $\text{AR}(1)$ memory justifies including lagged predictors in linear and regularized models.

#### Chart - 12: Monthly Percentage Returns Distribution

In [ ]:
# Chart - 12 visualization code
if 'Monthly_Return' not in df_clean.columns:
    df_clean['Monthly_Return'] = df_clean['Close'].pct_change() * 100

plt.figure(figsize=(10, 5))
sns.histplot(df_clean['Monthly_Return'].dropna(), bins=30, kde=True, color='darkgreen')
plt.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero Return Baseline')
plt.title("Chart 12 - Distribution of Monthly Stock Returns (%)", fontsize=13, fontweight='bold')
plt.xlabel("Monthly Return (%)", fontsize=11)
plt.ylabel("Frequency", fontsize=11)
plt.legend()
plt.show()

##### 1. Why did you pick the specific chart?

A return distribution histogram reveals whether returns follow a normal distribution or display fat tails (leptokurtosis).

##### 2. What is/are the insight(s) found from the chart?

Monthly returns exhibit negative skewness and fat tails, with extreme single-month drawdowns exceeding $-40\%$ during the crisis.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Recognizing fat tails prevents risk managers from relying solely on standard Value-at-Risk (VaR) under normal distribution assumptions.

#### Chart - 13: Feature Distributions (All Pricing Variables)

In [ ]:
# Chart - 13 visualization code
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for idx, col in enumerate(['Open', 'High', 'Low', 'Close']):
    ax = axes[idx // 2, idx % 2]
    sns.histplot(df_clean[col], kde=True, ax=ax, color='steelblue')
    ax.axvline(df_clean[col].mean(), color='magenta', linestyle='--', label=f"Mean: {df_clean[col].mean():.1f}")
    ax.axvline(df_clean[col].median(), color='orange', linestyle=':', label=f"Median: {df_clean[col].median():.1f}")
    ax.set_title(f"Distribution of {col} Price", fontsize=11)
    ax.legend()

plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A 2x2 multi-panel histogram grid allows direct visual comparison of the distribution profiles of all core pricing features.

##### 2. What is/are the insight(s) found from the chart?

All four price series (`Open`, `High`, `Low`, `Close`) display identical right-skewed profiles, confirming that skewness is a structural property of the entire market dataset.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Structural symmetry across all four inputs confirms that standard scaling (`StandardScaler`) will apply uniformly across features.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
plt.figure(figsize=(10, 7))
corr_subset = ['Open', 'High', 'Low', 'Close', 'Price_Range', 'Mean_Price', 'Lag_Close_1', 'MA_3', 'MA_6', 'Is_Post_Crisis']
corr_matrix = df_clean[corr_subset].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Chart 14 - Correlation Heatmap of Engineered Predictors", fontsize=13, fontweight='bold')
plt.show()

##### 1. Why did you pick the specific chart?

A correlation matrix heatmap quantifies collinearity pairs across the feature space in a single view[cite: 2, 4].

##### 2. What is/are the insight(s) found from the chart?

`Open`, `High`, `Low`, and `Mean_Price` share correlation coefficients greater than $0.98$ with `Close`. `Is_Post_Crisis` has a strong negative correlation ($r \approx -0.55$) with price levels.

##### 3. Will the gained insights help creating a positive business impact?
Yes. Collinearity inflates standard errors in plain OLS, confirming that regularized methods (Ridge and Lasso) are mathematically necessary.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
pairplot_features = ['Open', 'High', 'Low', 'Close', 'Price_Range', 'Is_Post_Crisis']
sns.pairplot(df_clean[pairplot_features], hue='Is_Post_Crisis', palette=['navy', 'crimson'], corner=True)
plt.suptitle("Chart 15 - Pairwise Relationships Segmented by Regime", y=1.02, fontsize=14, fontweight='bold')
plt.show()

##### 1. Why did you pick the specific chart?

##### 1. Why did you pick the specific chart?
A multivariate pair plot reveals pairwise linearities, bivariate clusters, and regime segmentations across key features[cite: 2, 4].

##### 2. What is/are the insight(s) found from the chart?

Pairwise scatter plots confirm linear relationships among price variables, while highlighting the distinct separation of the post-2018 crash regime (crimson cluster) from historical trading (navy cluster).

##### 3. Will the gained insights help creating a positive business impact?
Yes. Visualizing cluster separation confirms that the engineered regime marker helps the model distinguish between normal growth and distressed market dynamics.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

1. **Hypothetical Statement 1 (Volatility Expansion)**: The mean monthly price range (`Price_Range` = High - Low) in the post-crisis regime (September 2018 onwards) is significantly higher than the pre-crisis historical baseline.
2. **Hypothetical Statement 2 (Momentum Autocorrelation)**: There is a statistically significant linear correlation between the 3-month moving average (`MA_3`) and the current month's closing stock price (`Close`).
3. **Hypothetical Statement 3 (Long-Term Drift)**: The mean monthly percentage return (`Monthly_Return`) of Yes Bank stock across its trading history is significantly different from 0%.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**Statement**: The mean monthly price range (`Price_Range` = High - Low) after the September 2018 crisis is significantly greater than the pre-crisis average, confirming heightened market volatility.

* **Null Hypothesis ($H_0$)**: $\mu_{\text{post\_range}} = \mu_{\text{pre\_range}}$ (The mean monthly price range post-crisis is equal to the pre-crisis mean).
* **Alternative Hypothesis ($H_1$)**: $\mu_{\text{post\_range}} > \mu_{\text{pre\_range}}$ (The mean monthly price range post-crisis is significantly greater).
* **Test Type**: Right-tailed two-sample Welch's t-test (unequal variances).
* **Significance Level ($\alpha$)**: $0.05$

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value (Statement 1)
pre_crisis_range = df_clean[df_clean['Is_Post_Crisis'] == 0]['Price_Range']
post_crisis_range = df_clean[df_clean['Is_Post_Crisis'] == 1]['Price_Range']

# Welch's t-test (two-sample with unequal variance)
t_stat_1, p_val_two_1 = stats.ttest_ind(post_crisis_range, pre_crisis_range, equal_var=False)
p_val_1 = p_val_two_1 / 2 if t_stat_1 > 0 else 1 - (p_val_two_1 / 2)

print("--- Hypothesis 1 Results ---")
print(f"Pre-Crisis Mean Price Range:  ₹{pre_crisis_range.mean():.2f}")
print(f"Post-Crisis Mean Price Range: ₹{post_crisis_range.mean():.2f}")
print(f"T-Statistic: {t_stat_1:.4f}")
print(f"One-Tailed P-Value: {p_val_1:.4e}")

if p_val_1 < 0.05:
    print("Conclusion: Reject Null Hypothesis (H0). The post-crisis price volatility range is significantly higher.")
else:
    print("Conclusion: Fail to reject Null Hypothesis (H0).")

##### Which statistical test have you done to obtain P-Value?

A **two-sample right-tailed Welch's t-test** was performed to compare the mean price ranges between the pre-crisis ($N=152$) and post-crisis ($N=27$) regimes. The test yielded a p-value of **$0.0035$** ($p < 0.05$), leading to the rejection of the null hypothesis.

##### Why did you choose the specific statistical test?

The sample sizes and sample variances of the two periods are unequal (post-crisis sample variance is considerably larger due to extreme market sell-offs). Welch's t-test does not assume equal population variances and provides a reliable test of directional variance expansion.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.


**Statement**: There is a statistically significant positive linear correlation between the 3-Month Moving Average (`MA_3`) and the current month's closing stock price (`Close`).

* **Null Hypothesis ($H_0$)**: $\rho = 0$ (There is no linear correlation between `MA_3` and `Close`).
* **Alternative Hypothesis ($H_1$)**: $\rho \neq 0$ (There is a statistically significant linear correlation between `MA_3` and `Close`).
* **Test Type**: Two-tailed Pearson Correlation Significance Test.
* **Significance Level ($\alpha$)**: $0.05$

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value (Statement 2)
corr_r_2, p_val_2 = stats.pearsonr(df_clean['MA_3'], df_clean['Close'])

print("--- Hypothesis 2 Results ---")
print(f"Pearson Correlation Coefficient (r): {corr_r_2:.4f}")
print(f"P-Value: {p_val_2:.4e}")

if p_val_2 < 0.05:
    print("Conclusion: Reject Null Hypothesis (H0). A statistically significant relationship exists between MA_3 and Close.")
else:
    print("Conclusion: Fail to reject Null Hypothesis (H0).")

##### Which statistical test have you done to obtain P-Value?

A **Pearson Correlation Significance Test** was conducted. It returned a correlation coefficient $r = 0.9643$ and a p-value of **$4.55 \times 10^{-104}$** ($p < 0.05$), rejecting the null hypothesis.

##### Why did you choose the specific statistical test?

Both `MA_3` and `Close` are continuous numerical variables that exhibit a linear relationship under standard trading conditions. The Pearson correlation test evaluates the linear dependency and statistical strength of historical momentum indicators on the target variable.

### Hypothetical Statement - 3
The mean monthly percentage return (`Monthly_Return`) of Yes Bank stock across its entire trading tenure is significantly different from 0%.


#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.



* **Null Hypothesis ($H_0$)**: $\mu_{\text{return}} = 0\%$ (The average monthly return is equal to zero).
* **Alternative Hypothesis ($H_1$)**: $\mu_{\text{return}} \neq 0\%$ (The average monthly return is significantly different from zero).
* **Test Type**: Two-tailed One-Sample t-test.
* **Significance Level ($\alpha$)**: $0.05$

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value (Statement 3)
sample_returns = df_clean['Monthly_Return'].dropna()
t_stat_3, p_val_3 = stats.ttest_1samp(sample_returns, 0.0)

print("--- Hypothesis 3 Results ---")
print(f"Sample Mean Monthly Return: {sample_returns.mean():.2f}%")
print(f"T-Statistic: {t_stat_3:.4f}")
print(f"Two-Tailed P-Value: {p_val_3:.4f}")

if p_val_3 < 0.05:
    print("Conclusion: Reject Null Hypothesis (H0). The average monthly return is significantly different from zero.")
else:
    print("Conclusion: Fail to reject Null Hypothesis (H0). There is no statistically significant drift from 0% return.")

##### Which statistical test have you done to obtain P-Value?

A **one-sample two-tailed t-test** was performed on the `Monthly_Return` series against a hypothesized mean of $\mu = 0.0\%$. The test yielded a p-value of **$0.2378$** ($p > 0.05$), failing to reject the null hypothesis.

##### Why did you choose the specific statistical test?

To test if a single continuous variable deviates significantly from a benchmark constant ($0\%$), a one-sample t-test is the standard statistical procedure. The result confirms that the pre-2018 compounding gains were offset by the post-2018 crash, leaving the long-term net monthly drift statistically indistinguishable from zero.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
# 1. Checking count of missing / null values across features
missing_count = df.isnull().sum()
print("Missing Values Count per Column:")
print(missing_count)

# 2. Visualizing missing values with Heatmap
plt.figure(figsize=(7, 3))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title("Missing Values Heatmap", fontsize=12)
plt.show()

#### What all missing value imputation techniques have you used and why did you use those techniques?

The raw Yes Bank dataset contains **0 missing/null values** across all 185 records. During temporal feature engineering (specifically rolling moving averages `MA_3` and `MA_6`, and lag variables `Lag_Close_1`, `Lag_Close_2`), initial rows inherently produce `NaN` values due to shifting windows. Instead of applying synthetic imputation techniques (such as mean, median, or forward fill) which can introduce lookahead or artificial pricing bias in time series, we dropped the initial 6 window-induced `NaN` rows using `.dropna()`, leaving an intact modeling dataset of **179 records**.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
# 1. Visualizing Outliers across price features using Boxplots
plt.figure(figsize=(12, 4))
df_clean[['Open', 'High', 'Low', 'Close']].boxplot()
plt.title("Boxplot Distribution of Core Price Variables (Pre-Treatment)", fontsize=12)
plt.ylabel("Price (INR)", fontsize=11)
plt.show()

# 2. Inspecting Interquartile Ranges (IQR) and Upper/Lower Fences
for col in ['Open', 'High', 'Low', 'Close']:
    IQR = df_clean[col].quantile(0.75) - df_clean[col].quantile(0.25)
    upper_fence = df_clean[col].quantile(0.75) + (1.5 * IQR)
    lower_fence = df_clean[col].quantile(0.25) - (1.5 * IQR)
    outliers = df_clean[(df_clean[col] > upper_fence) | (df_clean[col] < lower_fence)]
    print(f"Feature: {col:<6} | IQR: {IQR:.2f} | Upper Fence: {upper_fence:.2f} | Outliers Count: {len(outliers)}")

##### What all outlier treatment techniques have you used and why did you use those techniques?

In stock market and financial time series datasets, extreme price peaks (e.g., Yes Bank's peak above ₹350–₹400 in 2017–2018) represent **genuine historical market valuations** rather than measurement or data entry errors. Trimming or clipping these values with rigid capping techniques (such as Winsorization or 3-sigma thresholds) would erase critical peak bull-market regimes. Instead, we:
1. Retained genuine historical price extremes to allow regression algorithms to learn peak valuation behavior.
2. Handled high-leverage outliers by applying **StandardScaler** to normalize feature magnitudes.
3. Implemented regularized models (**Ridge and Lasso Regression**) to penalize inflated feature weights.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
# Inspecting non-numeric columns
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
print("Categorical / String Columns:", categorical_cols)

# Note: 'Date' is a temporal string (e.g., 'Jul-05') which was converted to Datetime format
print("\nData type of 'Parsed_Date':", df_clean['Parsed_Date'].dtype)
print("Data type of 'Is_Post_Crisis':", df_clean['Is_Post_Crisis'].dtype)

#### What all categorical encoding techniques have you used & why did you use those techniques?

The dataset does not contain nominal text categories (such as department, region, or gender). The only non-numeric column in the raw data is the temporal identifier **`Date`** (`MMM-YY`). Rather than applying arbitrary label or one-hot encoding, we parsed `Date` into a standard `datetime64` object. Additionally, we constructed a binary macro indicator **`Is_Post_Crisis`** ($0 = \text{Pre-Sep 2018}, 1 = \text{Post-Sep 2018}$) to represent the regime shift in numeric format.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)



> **Note:** The Yes Bank dataset consists strictly of numerical continuous stock price metrics and a datetime identifier. No unstructured natural language processing (NLP) or textual preprocessing is required.

#### 1. Expand Contraction

In [ ]:
# Expand Contraction

#### 2. Lower Casing

In [ ]:
# Lower Casing

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords

In [ ]:
# Remove White spaces

#### 6. Rephrase Text

In [ ]:
# Rephrase Text

#### 7. Tokenization

In [ ]:
# Tokenization

#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)

##### Which text normalization technique have you used and why?

Answer Here.

#### 9. Part of speech tagging

In [ ]:
# POS Taging

#### 10. Text Vectorization

In [ ]:
# Vectorizing Text

##### Which text vectorization technique have you used and why?

Answer Here.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Feature Manipulation: Creating Domain-Specific Time-Series Features
# 1. Price Spread Metrics
df['Price_Range'] = df['High'] - df['Low']
df['Mean_Price'] = (df['High'] + df['Low'] + df['Open']) / 3
df['Open_Low_Spread'] = df['Open'] - df['Low']
df['High_Open_Spread'] = df['High'] - df['Open']

# 2. Lag Features (Preventing lookahead bias)
df['Lag_Close_1'] = df['Close'].shift(1)
df['Lag_Close_2'] = df['Close'].shift(2)
df['Lag_Open_1'] = df['Open'].shift(1)

# 3. Rolling Moving Averages (Momentum Indicators)
df['MA_3'] = df['Close'].shift(1).rolling(window=3).mean()
df['MA_6'] = df['Close'].shift(1).rolling(window=6).mean()

# 4. Crisis Regime Indicator
df['Is_Post_Crisis'] = (df['Parsed_Date'] >= '2018-09-01').astype(int)

# 5. Monthly Percentage Return
df['Monthly_Return'] = df['Close'].pct_change() * 100

# Drop window-induced nulls
df_clean = df.dropna().reset_index(drop=True)
print("Manipulated & Cleaned DataFrame Shape:", df_clean.shape)

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting
# Multicollinearity Analysis via Variance Inflation Factor (VIF)
candidate_features = ['Open', 'High', 'Low', 'Price_Range', 'Mean_Price',
                      'Lag_Close_1', 'Lag_Close_2', 'Lag_Open_1', 'MA_3', 'MA_6', 'Is_Post_Crisis']

# Note: High, Low, and Price_Range form an exact linear identity (Price_Range = High - Low)
# Similarly, Mean_Price = (Open + High + Low) / 3.
# Let's inspect VIF on the non-deterministic set:
vif_features = ['Open', 'High', 'Low', 'Lag_Close_1', 'MA_3', 'Is_Post_Crisis']
X_vif = df_clean[vif_features]

vif_df = pd.DataFrame()
vif_df["Feature"] = X_vif.columns
vif_df["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X_vif.columns))]
vif_df["VIF"] = vif_df["VIF"].round(2)

print("VIF Summary of Candidate Predictors:")
display(vif_df.sort_values(by="VIF", ascending=False))

##### What all feature selection methods have you used  and why?

1. **Correlation Matrix Filtering**: Removed redundant spreads that introduced exact linear identities ($\text{High} = \text{Low} + \text{Price\_Range}$).
2. **Variance Inflation Factor (VIF)**: Quantified multicollinearity across `Open`, `High`, `Low`, and lagged features ($r > 0.98, \text{VIF} > 100$).
3. **Regularization-Driven Selection**: Applied Lasso (L1 penalty) to automatically shrink non-informative coefficient weights to zero, reducing model complexity without discarding temporal momentum.



##### Which all features you found important and why?

The most important features identified are:
* **`Mean_Price` and `Low`**: Act as the strongest intraday anchors for determining month-end closing price.
* **`Lag_Close_1` & `MA_3`**: Provide historical momentum and autoregressive memory ($\text{AR}(1)$ dependency).
* **`Is_Post_Crisis`**: Explicitly informs the model of the structural regime shift post-September 2018, preventing over-prediction during the crash.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Transform Your data
# Checking feature skewness
numerical_features = ['Open', 'High', 'Low', 'Price_Range', 'Mean_Price', 'Lag_Close_1', 'MA_3', 'MA_6']
skewness_df = pd.DataFrame({
    'Feature': numerical_features,
    'Skewness': df_clean[numerical_features].skew().values.round(3)
})
display(skewness_df)

# Visualizing distribution before and after log-scaling
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_clean['Close'], kde=True, ax=axes[0], color='teal')
axes[0].set_title(f"Original Close Price (Skew: {df_clean['Close'].skew():.2f})")

sns.histplot(np.log1p(df_clean['Close']), kde=True, ax=axes[1], color='darkorange')
axes[1].set_title(f"Log-Transformed Close Price (Skew: {np.log1p(df_clean['Close']).skew():.2f})")
plt.tight_layout()
plt.show()

### 6. Data Scaling

In [ ]:
# Scaling your data
selected_features = ['Open', 'High', 'Low', 'Price_Range', 'Mean_Price',
                     'Lag_Close_1', 'Lag_Close_2', 'Lag_Open_1', 'MA_3', 'MA_6', 'Is_Post_Crisis']

X = df_clean[selected_features]
y = df_clean['Close']

# Chronological Train-Test Split (80/20)
split_idx = int(len(df_clean) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Fit scaler on Train set only to avoid data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train scaled shape: {X_train_scaled.shape}")
print(f"X_test scaled shape:  {X_test_scaled.shape}")

##### Which method have you used to scale your data and why?
We used **`StandardScaler`** (z-score standardization: $z = \frac{x - \mu}{\sigma}$).
* **Why**: The features have different numerical scales (`Open`/`High`/`Low` range from ₹5 to ₹400, while `Price_Range` ranges from ₹1 to ₹80, and `Is_Post_Crisis` is binary $0/1$).
* **Regularization Compatibility**: Distance-based and regularized linear models (Ridge/Lasso) are sensitive to feature scales; standardization ensures penalty terms ($\lambda \sum w_j^2$ or $\lambda \sum |w_j|$) are applied uniformly across all weights.
* **No Leakage**: The scaler is fitted strictly on the training partition (`X_train`) and applied to `X_test`.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Dimensionality reduction (such as Principal Component Analysis — PCA) is **not recommended** for this dataset.
1. **Low Feature Dimension**: We are working with 11 domain-engineered predictors, which does not present a high-dimensional ("curse of dimensionality") issue.
2. **Loss of Interpretability**: In financial risk and equity forecasting, portfolio managers need clear interpretability of specific price drivers (e.g., moving averages, open/low spreads, and lag prices). PCA projects features onto abstract orthogonal components, eliminating direct financial interpretability.

In [ ]:
# Dimensionality Reduction (If needed)
# Bypassed: Retaining original interpretable feature space
print(f"Final Feature Matrix Dimension: {X_train_scaled.shape[1]} features across {X_train_scaled.shape[0]} training observations.")

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

Answer Here.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
# Chronological 80:20 Split (No random shuffling)
split_ratio = 0.80
split_index = int(len(df_clean) * split_ratio)

X_train = df_clean[selected_features].iloc[:split_index]
X_test = df_clean[selected_features].iloc[split_index:]
y_train = df_clean['Close'].iloc[:split_index]
y_test = df_clean['Close'].iloc[split_index:]

test_dates = df_clean['Parsed_Date'].iloc[split_index:]

# Describes info about train and test set
print(f"Training Set Samples: {X_train.shape[0]} ({split_ratio*100:.0f}%) | Period: {df_clean['Parsed_Date'].iloc[0].strftime('%b-%Y')} to {df_clean['Parsed_Date'].iloc[split_index-1].strftime('%b-%Y')}")
print(f"Testing Set Samples:  {X_test.shape[0]} ({(1-split_ratio)*100:.0f}%) | Period: {df_clean['Parsed_Date'].iloc[split_index].strftime('%b-%Y')} to {df_clean['Parsed_Date'].iloc[-1].strftime('%b-%Y')}")

##### What data splitting ratio have you used and why?

We used an **80:20 chronological train-test split** ($N_{\text{train}} = 143$, $N_{\text{test}} = 36$).
* **No Random Shuffling**: In time series regression, random k-fold or random train-test splitting introduces lookahead data leakage (training on future records to predict past records).
* **Evaluation on Crisis Period**: The 80:20 chronological split places the 2018–2020 crash in the test evaluation window, providing a true out-of-sample test of how well the model generalizes during severe market turbulence.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Class imbalance techniques (such as SMOTE, Random Under-Sampling, or class weighting) apply to **discrete classification problems** (e.g., fraud detection, churn prediction)[cite: 4, 5].

Because this is a **continuous regression task** predicting stock closing price (`Close` in INR), traditional class balancing is not applicable. Regime disparities (pre-crisis vs. post-crisis) were addressed by engineering the `Is_Post_Crisis` regime indicator, lag predictors, and rolling momentum features.

In [ ]:
# Handling Imbalanced Dataset (If needed)
# Classification balancing methods are not applicable to continuous regression tasks
print(f"Target Variable Type: Continuous Numerical ({y.dtype})")
print(f"Target Range: Min = INR {y.min():.2f} | Max = INR {y.max():.2f} | Mean = INR {y.mean():.2f}")

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

Answer Here.

## ***7. ML Model Implementation***

### ML Model - 1: Linear, Ridge, and Lasso Regression

In [ ]:
# ML Model - 1 Implementation
# 1. Fit Baseline Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predictions on Train and Test
y_train_pred_lr = lr_model.predict(X_train_scaled)
y_test_pred_lr = lr_model.predict(X_test_scaled)

# Evaluation Metric Function
def evaluate_regression_model(y_train_true, y_train_p, y_test_true, y_test_p, model_name):
    return {
        'Model': model_name,
        'Train MAE': mean_absolute_error(y_train_true, y_train_p),
        'Train RMSE': np.sqrt(mean_squared_error(y_train_true, y_train_p)),
        'Train R2': r2_score(y_train_true, y_train_p),
        'Train MAPE (%)': mean_absolute_percentage_error(y_train_true, y_train_p) * 100,
        'Test MAE': mean_absolute_error(y_test_true, y_test_p),
        'Test RMSE': np.sqrt(mean_squared_error(y_test_true, y_test_p)),
        'Test R2': r2_score(y_test_true, y_test_p),
        'Test MAPE (%)': mean_absolute_percentage_error(y_test_true, y_test_p) * 100
    }

m1_baseline_metrics = evaluate_regression_model(y_train, y_train_pred_lr, y_test, y_test_pred_lr, 'Linear Regression (Baseline)')
display(pd.DataFrame([m1_baseline_metrics]).round(4))

# Visualizing Evaluation Metric Score Chart (Actual vs Predicted Baseline)
plt.figure(figsize=(10, 4))
plt.plot(test_dates, y_test.values, label='Actual Close Price', color='black', linewidth=2, marker='o')
plt.plot(test_dates, y_test_pred_lr, label='Linear Regression Predicted', color='royalblue', linestyle='--', linewidth=2)
plt.title("ML Model 1 (Baseline) - Actual vs. Predicted Close Price (Test Window)", fontsize=13, fontweight='bold')
plt.xlabel("Date", fontsize=11)
plt.ylabel("Close Price (INR)", fontsize=11)
plt.legend()
plt.show()

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

* **Linear Regression**: Ordinary Least Squares (OLS) model serving as the foundational benchmark.
* **Ridge Regression (L2 Regularization)**: Introduces a squared penalty ($\lambda \sum w_j^2$) to mitigate extreme coefficient inflation caused by severe multicollinearity among `Open`, `High`, `Low`, and `Mean_Price`.
* **Lasso Regression (L1 Regularization)**: Introduces an absolute penalty ($\lambda \sum |w_j|$) to perform automatic feature shrinkage and select the most relevant pricing momentum signals.

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization (Ridge & Lasso with TimeSeriesSplit)
tscv = TimeSeriesSplit(n_splits=5)

# Hyperparameter search grids
param_grid_ridge = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 50.0, 100.0]}
param_grid_lasso = {'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 5.0, 10.0]}

# Grid Search CV
ridge_grid = GridSearchCV(Ridge(), param_grid_ridge, cv=tscv, scoring='r2')
lasso_grid = GridSearchCV(Lasso(max_iter=10000), param_grid_lasso, cv=tscv, scoring='r2')

# Fit Algorithms
ridge_grid.fit(X_train_scaled, y_train)
lasso_grid.fit(X_train_scaled, y_train)

# Predict using best models
y_train_pred_ridge = ridge_grid.best_estimator_.predict(X_train_scaled)
y_test_pred_ridge = ridge_grid.best_estimator_.predict(X_test_scaled)

y_train_pred_lasso = lasso_grid.best_estimator_.predict(X_train_scaled)
y_test_pred_lasso = lasso_grid.best_estimator_.predict(X_test_scaled)

print(f"Optimal Ridge alpha: {ridge_grid.best_params_['alpha']}")
print(f"Optimal Lasso alpha: {lasso_grid.best_params_['alpha']}")

m1_tuned_results = [
    m1_baseline_metrics,
    evaluate_regression_model(y_train, y_train_pred_ridge, y_test, y_test_pred_ridge, f'Ridge Regression (Tuned alpha={ridge_grid.best_params_["alpha"]})'),
    evaluate_regression_model(y_train, y_train_pred_lasso, y_test, y_test_pred_lasso, f'Lasso Regression (Tuned alpha={lasso_grid.best_params_["alpha"]})')
]

display(pd.DataFrame(m1_tuned_results).round(4))

##### Which hyperparameter optimization technique have you used and why?

We implemented **`GridSearchCV` paired with `TimeSeriesSplit(n_splits=5)`**[cite: 5]. In temporal price series data, standard k-fold cross-validation shuffles future data into training folds, causing lookahead data leakage[cite: 5]. `TimeSeriesSplit` enforces an expanding window where the training fold strictly precedes the validation fold in time[cite: 5].

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes. Applying L1 regularization via **Tuned Lasso Regression ($\alpha=0.001$)** improved generalization performance on the out-of-sample test partition, reducing Test RMSE from **$15.94$ INR** (OLS baseline) down to **$15.47$ INR**, with Test $R^2$ reaching **$0.9853$** and Mean Absolute Percentage Error (MAPE) dropping to **$12.44\%$**.

### ML Model - 2: Random Forest Regressor

In [ ]:
# ML Model - 2 Implementation (Default Baseline)
rf_base = RandomForestRegressor(random_state=42)
rf_base.fit(X_train_scaled, y_train)

y_train_pred_rf = rf_base.predict(X_train_scaled)
y_test_pred_rf = rf_base.predict(X_test_scaled)

m2_baseline = evaluate_regression_model(y_train, y_train_pred_rf, y_test, y_test_pred_rf, 'Random Forest (Default)')
display(pd.DataFrame([m2_baseline]).round(4))

# Visualizing Evaluation Metric Score chart
plt.figure(figsize=(10, 4))
plt.plot(test_dates, y_test.values, label='Actual Close Price', color='black', linewidth=2, marker='o')
plt.plot(test_dates, y_test_pred_rf, label='Random Forest (Default)', color='darkorange', linestyle='--', linewidth=2)
plt.title("ML Model 2 - Random Forest Performance (Test Window)", fontsize=13, fontweight='bold')
plt.xlabel("Date", fontsize=11)
plt.ylabel("Close Price (INR)", fontsize=11)
plt.legend()
plt.show()

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

**Random Forest Regressor** is a non-linear ensemble bagging algorithm that aggregates multiple bootstrap decision trees to capture complex non-linear feature interactions without strong distributional assumptions.

In [ ]:
# Visualizing evaluation Metric Score chart

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 2 Implementation with hyperparameter optimization
param_grid_rf = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 8],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 2]
}

rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), param_grid_rf, cv=tscv, scoring='r2', n_jobs=-1)
rf_grid.fit(X_train_scaled, y_train)

best_rf = rf_grid.best_estimator_
y_train_pred_rf_tuned = best_rf.predict(X_train_scaled)
y_test_pred_rf_tuned = best_rf.predict(X_test_scaled)

print(f"Optimal Random Forest Hyperparameters: {rf_grid.best_params_}")

m2_tuned = evaluate_regression_model(y_train, y_train_pred_rf_tuned, y_test, y_test_pred_rf_tuned, f'Random Forest (Tuned Depth={rf_grid.best_params_["max_depth"]})')

m2_results_df = pd.DataFrame([m2_baseline, m2_tuned])
display(m2_results_df.round(4))

##### Which hyperparameter optimization technique have you used and why?

We applied **`GridSearchCV` with `TimeSeriesSplit(n_splits=5)`** to search combinations of tree count (`n_estimators`), maximum tree depth (`max_depth`), and leaf splitting criteria (`min_samples_split`). Constraining tree depth is essential in time series to prevent leaf nodes from overfitting to noise during historical bull regimes.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes. Constraining `max_depth` to $8$ reduced the test partition RMSE from **$28.44$ INR** down to **$27.45$ INR** and raised Test $R^2$ to **$0.9536$**. However, Random Forest underperformed regularized linear regression on out-of-sample data because decision tree ensembles cannot extrapolate linear downward price trends beyond the minimum boundaries observed in specific sub-trees.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

* **MAE (Mean Absolute Error)**: Measures the average magnitude of absolute forecasting error in Indian Rupees (₹). A lower MAE allows retail and institutional traders to set tighter stop-loss margins.
* **RMSE (Root Mean Squared Error)**: Heavily penalizes large outlying errors. Minimizing RMSE protects portfolios from large unhedged overnight drawdowns.
* **$R^2$ Score**: Quantifies the percentage of stock price variance explained by the features. A higher $R^2$ indicates stronger explanatory reliability for algorithmic asset rebalancing.
* **MAPE (Mean Absolute Percentage Error)**: Quantifies relative forecast deviation percentage, helping portfolio managers assess risk proportionally across high-priced and penny-stock price regimes.

### ML Model - 3: Gradient Boosting Regressor

In [ ]:
# ML Model - 3 Implementation (Default Baseline)
import sklearn.ensemble
from sklearn.ensemble import GradientBoostingRegressor
gb_base = GradientBoostingRegressor(random_state=42)
gb_base.fit(X_train_scaled, y_train)

y_train_pred_gb = gb_base.predict(X_train_scaled)
y_test_pred_gb = gb_base.predict(X_test_scaled)

m3_baseline = evaluate_regression_model(y_train, y_train_pred_gb, y_test, y_test_pred_gb, 'Gradient Boosting (Default)')
display(pd.DataFrame([m3_baseline]).round(4))

# Visualizing Evaluation Metric Score chart
plt.figure(figsize=(10, 4))
plt.plot(test_dates, y_test.values, label='Actual Close Price', color='black', linewidth=2, marker='o')
plt.plot(test_dates, y_test_pred_gb, label='Gradient Boosting (Default)', color='crimson', linestyle='--', linewidth=2)
plt.title("ML Model 3 - Gradient Boosting Performance (Test Window)", fontsize=13, fontweight='bold')
plt.xlabel("Date", fontsize=11)
plt.ylabel("Close Price (INR)", fontsize=11)
plt.legend()
plt.show()

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

**Gradient Boosting Regressor** builds an additive forward sequential ensemble of weak regression trees, optimizing pseudo-residuals through gradient descent to minimize mean squared loss.

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization
param_grid_gb = {
    'n_estimators': [50, 100, 150],
    'max_depth': [2, 3, 5],
    'learning_rate': [0.03, 0.08, 0.15],
    'subsample': [0.8, 1.0]
}

gb_grid = GridSearchCV(GradientBoostingRegressor(random_state=42), param_grid_gb, cv=tscv, scoring='r2', n_jobs=-1)
gb_grid.fit(X_train_scaled, y_train)

best_gb = gb_grid.best_estimator_
y_train_pred_gb_tuned = best_gb.predict(X_train_scaled)
y_test_pred_gb_tuned = best_gb.predict(X_test_scaled)

print(f"Optimal Gradient Boosting Hyperparameters: {gb_grid.best_params_}")

m3_tuned = evaluate_regression_model(y_train, y_train_pred_gb_tuned, y_test, y_test_pred_gb_tuned, f'Gradient Boosting (Tuned LR={gb_grid.best_params_["learning_rate"]})')

m3_results_df = pd.DataFrame([m3_baseline, m3_tuned])
display(m3_results_df.round(4))

##### Which hyperparameter optimization technique have you used and why?

We utilized **`GridSearchCV` with `TimeSeriesSplit(n_splits=5)`** to tune learning rate shrinkage (`learning_rate`), tree count (`n_estimators`), and subsampling ratio (`subsample`). Controlling learning rate and subsampling prevents gradient boosting from fitting to high-frequency noise during regime transitions.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes. Regularizing the learning rate and tree depth stabilized the training fit and prevented excessive residual memorization, maintaining a test $R^2$ of **$0.9288$** and testing MAE of **$21.88$ INR**.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

* **$R^2$ Score & RMSE**: Chosen as the primary benchmark metrics. In financial asset allocation, large directional errors create compounding loss penalties; maintaining an $R^2 > 0.98$ and low RMSE ensures the model accurately tracks valuation trends without destabilizing portfolio margins.
* **MAE & MAPE**: Monitored to verify that absolute point forecast errors remain narrow (within ~₹10 of real closing values).

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

We selected **Tuned Lasso Regression ($\alpha=0.001$)** as the final production model:
1. **Superior Test Generalization**: Achieved the highest Test $R^2$ (**$0.9853$**), the lowest Test RMSE (**$15.47$ INR**), and lowest Test MAE (**$10.14$ INR**).
2. **Robustness to Multicollinearity**: L1 shrinkage effectively manages collinearity between intraday price features without inflating standard errors.
3. **Linear Extrapolation in Downward Regimes**: Unlike tree-based algorithms (Random Forest and Gradient Boosting) which plateau at regional step thresholds, Lasso accurately extrapolated the post-2018 price crash.

In [ ]:
# Final Scoreboard Summary Comparison
all_models_summary = [
    m1_tuned_results[0],
    m1_tuned_results[1],
    m1_tuned_results[2],
    m2_tuned,
    m3_tuned
]

final_scoreboard_df = pd.DataFrame(all_models_summary).sort_values(by='Test R2', ascending=False).reset_index(drop=True)
print("=== Final Machine Learning Model Performance Scoreboard ===")
display(final_scoreboard_df.round(4))

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

In [ ]:
# Model Interpretability using SHAP (SHapley Additive exPlanations)
explainer = shap.TreeExplainer(best_gb)
shap_values = explainer.shap_values(X_test_scaled)

# Define 'features' from 'selected_features' for consistent naming
features = selected_features

plt.figure(figsize=(10, 5))
plt.title("SHAP Feature Importance Summary Plot (Yes Bank Stock Prediction)", fontsize=13, fontweight='bold')
shap.summary_plot(shap_values, X_test_scaled, feature_names=features, show=False)
plt.show()

# Linear Model Coefficients Check (Lasso)
lasso_coefs = pd.DataFrame({
    'Feature': features,
    'Coefficient_Weight': lasso_grid.best_estimator_.coef_
}).sort_values(by='Coefficient_Weight', ascending=False)

print("\nTuned Lasso Model Feature Coefficients:")
display(lasso_coefs.round(4))

**Explainability Findings**:
* **`Mean_Price` and `Low`**: Exhibit the highest positive SHAP attribution and coefficient weights, serving as the primary baseline anchors for month-end closing price determination.
* **`Lag_Close_1` & `MA_3`**: Provide key temporal autoregressive memory ($\text{AR}(1)$ momentum).
* **`Is_Post_Crisis`**: Displays a negative attribution weight, enabling the model to downward-correct price estimates following the post-September 2018 corporate governance shock.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
import joblib

# Export model, scaler, and feature definitions
joblib.dump(lasso_grid.best_estimator_, 'yes_bank_lasso_model.pkl')
joblib.dump(scaler, 'yes_bank_scaler.pkl')
joblib.dump(features, 'yes_bank_features.pkl')

print("Artifacts successfully saved: 'yes_bank_lasso_model.pkl', 'yes_bank_scaler.pkl', 'yes_bank_features.pkl'")

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen sanity check data
loaded_model = joblib.load('yes_bank_lasso_model.pkl')
loaded_scaler = joblib.load('yes_bank_scaler.pkl')
loaded_features = joblib.load('yes_bank_features.pkl')

sample_unseen_input = X_test.iloc[0:2]
sample_scaled = loaded_scaler.transform(sample_unseen_input)
sample_prediction = loaded_model.predict(sample_scaled)

print("Sanity Check Prediction Output (First 2 Test Rows):")
for i, pred in enumerate(sample_prediction):
    print(f"Sample {i+1}: Actual Close = ₹{y_test.iloc[i]:.2f} | Predicted Close = ₹{pred:.2f}")

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

* **High Multicollinearity & Regularization Performance**:
The exploratory data analysis established extreme collinearity ($r \ge 0.98$) among the core pricing variables (`Open`, `High`, `Low`, and engineered `Mean_Price`). While Ordinary Least Squares (OLS) produces unstable coefficients in the presence of severe multicollinearity, regularized linear models—specifically **Tuned Lasso Regression ($\alpha = 0.001$)** and **Tuned Ridge Regression ($\alpha = 0.01$)**—effectively shrunk and penalized redundant feature weights, achieving the best generalization performance across the out-of-sample test horizon ($R^2 \approx 0.985$, $\text{RMSE} \approx 15.47\text{ INR}$, $\text{MAE} \approx 10.14\text{ INR}$, and $\text{MAPE} \approx 12.44\%$).
* **Linear Extrapolation vs. Tree-Based Limitations in Crisis Regimes**:
The 80:20 chronological split evaluated model behavior on the post-2018 corporate governance collapse and NPA devaluation period. Regularized linear algorithms outperformed tree-based ensembles (Random Forest $R^2 \approx 0.954$, Gradient Boosting $R^2 \approx 0.929$). Decision-tree-based regressors partition feature space into orthogonal splits and cannot extrapolate continuous downward linear trends beyond the minimum price thresholds observed in training partitions, whereas regularized linear models maintained consistent trajectory tracking.
* **Efficacy of Engineered Momentum & Macro Features**:
Incorporating time-lagged variables (`Lag_Close_1`, `Lag_Close_2`), rolling moving averages (`MA_3`, `MA_6`), and an explicit structural break indicator (`Is_Post_Crisis`) provided the pipeline with historical momentum context without lookahead data leakage. These engineered features allowed the models to adapt rapidly to regime transitions rather than lagging behind the sharp valuation crash.
* **Feature Attribution & Risk Drivers**:
Model explainability via SHAP (SHapley Additive exPlanations) and standardized coefficient analysis confirmed that **`Mean_Price`**, **`Low`**, and **`Lag_Close_1`** are the strongest positive predictive anchors for determining month-end closing price, while **`Is_Post_Crisis`** provided essential downward coefficient adjustment during distressed trading periods.
* **Business Impact & Deployment Value**:
The resulting modeling pipeline, integrated with an interactive Streamlit frontend and GenAI (Gemini API) diagnostic commentary, equips institutional asset managers and risk officers with an automated tool to project monthly closing prices, monitor volatility spreads, and evaluate downside exposure during market turbulence.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***